In [3]:
import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, LeaveOneGroupOut
from sklearn.metrics import f1_score

# 1. Mount and Load
drive.mount('/content/drive')
dataset_path = '/content/drive/MyDrive/Colab/windowed.csv'
df = pd.read_csv(dataset_path)

X = df.drop(columns=['class', 'user'])
y = df['class']
groups = df['user']
user_list = groups.unique()

# 2. Setup Hyperparameter Space
param_dist_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'bootstrap': [True]
}

# 3. Manual Nested LOSO Loop
outer_loso = LeaveOneGroupOut()
nested_f1_scores = []

print(f"--- Starting Manual Nested LOSO (Inner & Outer) ---")
print(f"Total Users: {len(user_list)}\n")

# The Outer Loop: Leaving one user out for testing
for fold, (train_idx, test_idx) in enumerate(outer_loso.split(X, y, groups)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    groups_train = groups.iloc[train_idx]

    held_out_user = groups.iloc[test_idx].unique()[0]
    print(f"Fold {fold+1}: Testing on user [{held_out_user}]")
    print(f"  - Training on users: {groups_train.unique().tolist()}")

    # The Inner Loop: RandomizedSearch using LOSO on the remaining users
    inner_loso = LeaveOneGroupOut()
    rf_classifier = RandomForestClassifier(random_state=42, n_jobs=-1)

    inner_search = RandomizedSearchCV(
        estimator=rf_classifier,
        param_distributions=param_dist_rf,
        n_iter=1,
        cv=inner_loso,
        scoring='f1_weighted',
        n_jobs=-1,
        random_state=42
    )

    # CRITICAL FIX: We pass groups_train here so the inner LOSO knows the subjects
    inner_search.fit(X_train, y_train, groups=groups_train)

    # Best model for this outer fold
    best_model = inner_search.best_estimator_
    print(f"  - Best Params: {inner_search.best_params_}")

    # Evaluate on the held-out user
    y_pred = best_model.predict(X_test)
    score = f1_score(y_test, y_pred, average='weighted')
    nested_f1_scores.append(score)
    print(f"  - F1-score for this fold: {score:.4f}\n")

# 4. Final Results
print("="*50)
print("FINAL NESTED LOSO PERFORMANCE")
print("="*50)
for i, score in enumerate(nested_f1_scores):
    print(f"User {user_list[i]}: {score:.4f}")

print("-" * 50)
print(f"Mean F1-score: {np.mean(nested_f1_scores):.4f}")
print(f"Std Deviation: {np.std(nested_f1_scores):.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- Starting Manual Nested LOSO (Inner & Outer) ---
Total Users: 4

Fold 1: Testing on user [debora]
  - Training on users: ['katia', 'wallace', 'jose_carlos']
  - Best Params: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': 20, 'bootstrap': True}
  - F1-score for this fold: 0.8915

Fold 2: Testing on user [jose_carlos]
  - Training on users: ['debora', 'katia', 'wallace']
  - Best Params: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': 20, 'bootstrap': True}
  - F1-score for this fold: 0.9140

Fold 3: Testing on user [katia]
  - Training on users: ['debora', 'wallace', 'jose_carlos']
  - Best Params: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': 20, 'bootstrap': True}
  - F1-score for this fold: 0.7084

Fold 4: Testing on user [wallace]
  - Training on use